## Scraping Agents Workflow Architecture Prototyping 

The main agents that will be tested are going to be: 
- Item Selection Agent: chooses which listings may be valid and worth investigating, selects like 3-9 listings, and then 1-3 are chosen and the rest are used as backup in case the listings researched were invalid.
- Orchestrator Agent: When given a listing deemed worthy of investigating, coordinates and monitors spawned sub-agents, collects their findings and flagged ambiguities, and delivers a final buy/watch/ignore verdict with reasoning.
- eBay Sub-Agent: Searches eBay for sold listings matching the given item, normalizes results against the source listing (title, condition, colorway, year where available), flags ambiguous matches, and returns structured comp data to the orchestrator.

## Agent-To-Agent-Interactions 
Item Selection <--> Item Selection: 
- only interacts with itself, and once the listings are found, the data is stored and gone through sequentially, spawning one orchestrator agent at at time. 

Orchestrator -> eBay: 
- resolves eBay listing confusion
- monitors sub-agents(can pause, stop, or kill a sub-agent) 

eBay -> Orchestrator: 
- flags eBay listing and calls Orchestrator agent to help resolve confusion

# Agent Inputs 
Item Selection: 
- structured/parsed html from TheRealReal brand catalog page 

Orchestrator: 
- all info pertaining to a given listing, use schemas/models based on the category of the garment 
- eBay flagged listing info

eBay: 
- all info pertaining to a given listing, use schemas/models based on the category of the garment 

# Agent Outputs
- TODO

# Important Data Centralizations
- All item listing info is centralized into one structure that ALL agents will use to make context injection easier. 
- All listings regardless if they are found on eBay or TheRealReal all share the same model/schema based on the type of garment(shoes, tshirt, jacket, etc). 
- 


In [ ]:
from typing import TypedDict, Annotated                                                                          
  from langgraph.graph import StateGraph                                                                           
  import operator                                                                                                  
                                                                                                                   
  class AgentState(TypedDict):                                                                                     
      listing: dict                                                                                                
      ebay_comps: list                                                                                           
      flags: Annotated[list, operator.add]  # agents append, don't overwrite
      verdict: str 

In [ ]:
 def orchestrator_node(state: AgentState):
      # reason over state, return partial update
      return {"verdict": "buy"}

  def ebay_agent_node(state: AgentState):
      return {"ebay_comps": [...], "flags": ["ambiguous colorway"]}

# Item Selection Agent Prototype
Processes TRR listings in batches of 2, accumulates picks until quota (2) is met, then stops. Picks are stored in state to be handed off to the TRR detail agent.

In [ ]:
  from langgraph.graph import StateGraph, END                                                                      
  from typing import TypedDict                                                                                     
                                                                                                                   
  # 1. state — shared dict all nodes read/write                                                                    
  class State(TypedDict):                                                                                        
      input: str                                                                                                   
      output: str                                                                                                

  # 2. node — function that takes state, returns partial update                                                    
  def my_node(state: State):
      return {"output": "did something with: " + state["input"]}                                                   
                                                                                                                   
  # 3. wire it
  graph = StateGraph(State)                                                                                        
  graph.add_node("my_node", my_node)                                                                             
  graph.set_entry_point("my_node")
  graph.add_edge("my_node", END)
                                                                                                                   
  app = graph.compile()
                                                                                                                   
  # 4. run it                                                                                                    
  result = app.invoke({"input": "hello", "output": ""})
  print(result["output"])     